# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Loads the starter CSV and rebuilds the Week-4 baseline score so it can be evaluated on the *same* split as the model, later in this notebook.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Rebuild the Week-4 baseline score exactly, so it can be scored on the model's held-out split below.
tier_ctr = df[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].mean()
df["expected_ctr_tier"] = df["position_tier"].map(tier_ctr)
eligible = (df["impressions_90d"] >= 200) & df["expected_ctr_tier"].notna()
gap = (df["expected_ctr_tier"] - df["ctr"]).clip(lower=0)
df["baseline_score"] = np.where(eligible, gap * df["impressions_90d"], 0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"{len(df):,} rows | overall base rate: {df['is_declining_label'].mean():.3f}")

30,000 rows | overall base rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression first, then Random Forest** — per the toolkit's own mapping, this is a "which first?" ranking problem (a queue, evaluated at precision@K), and the recommended path for that shape is any classifier's probability output, read as a ranking score rather than a hard label.

I'm training both rather than jumping straight to the forest: logistic regression is the readable baseline-for-the-model (coefficients I can name and sanity-check), and the forest is added only if it earns its extra complexity on the comparison table in Section 3 — not by default. This mirrors notebook 02's depth-2-tree lesson: simplicity is worth defending until something beats it by a real margin, not a rounding error.

**Features** (all pre-decision, non-leaky — same discipline as w02/w03): `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`, `engagement_rate`. `trend_direction`/`trend_pct` are excluded as always — they're what the label is built from.

In [2]:
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "engagement_rate"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
print(f"{len(features)} features, {len(X):,} rows, {y.mean():.3f} overall positive rate")

7 features, 30,000 rows, 0.542 overall positive rate


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, 75/25.** A random row-level split would let the same client's pages sit in both train and test — the model could partly learn "which client is this" (its typical traffic level, niche, volatility) rather than "is this page declining," and the score would look better than it would on a genuinely new client. I already saw this exact gap open up in notebook 02 (in-sample vs. client-holdout flipped which method won at which K), so I'm not repeating that mistake here — grouped from the start.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
baseline_te = df["baseline_score"].values[test_idx]

print(f"train: {len(y_tr):,} rows, {df['client_id'].iloc[train_idx].nunique()} clients")
print(f"test:  {len(y_te):,} rows, {df['client_id'].iloc[test_idx].nunique()} clients")
print(f"test base rate: {y_te.mean():.3f}")
print("\nCaveat to carry forward: only 8 clients land in this test set (out of 32 total),")
print("so treat the precision numbers below as directional, not precise to the third decimal.")

train: 22,885 rows, 24 clients
test:  7,115 rows, 8 clients
test base rate: 0.517

Caveat to carry forward: only 8 clients land in this test set (out of 32 total),
so treat the precision numbers below as directional, not precise to the third decimal.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test rows (`test_idx`), same metric (precision@20, precision@50) as Week 4 — the baseline score is scored on this held-out split too, not the full dataset, so this is an apples-to-apples table. A `DummyClassifier` (stratified) is included as the floor below the floor, per the skill's suggestion.

In [4]:
logit = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
rf    = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                class_weight="balanced").fit(X_tr, y_tr)
dummy = DummyClassifier(strategy="stratified", random_state=42).fit(X_tr, y_tr)

scores = {
    "baseline (Week 4 rule)": baseline_te,
    "dummy (stratified)":     dummy.predict_proba(X_te)[:, 1],
    "logistic regression":    logit.predict_proba(X_te)[:, 1],
    "random forest":          rf.predict_proba(X_te)[:, 1],
}

rows = []
for name, s in scores.items():
    rows.append({
        "method": name,
        "precision@20": round(precision_at_k(s, y_te.values, 20), 3),
        "precision@50": round(precision_at_k(s, y_te.values, 50), 3),
    })
comparison = pd.DataFrame(rows)
comparison

,method,precision@20,precision@50
0,baseline (Week 4 rule),0.40,0.54
1,dummy (stratified),0.55,0.58
2,logistic regression,0.65,0.66
3,random forest,0.85,0.66


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the forest leans on, and does it make sense:** `impressions_90d` and `avg_position` dominate, followed by `content_age_days` and `word_count`; `days_since_last_update` is the weakest of the seven. That last part is consistent with Signal 1 from Week 4 coming back MIXED — the model independently arrived at the same conclusion the manual signal check did, which is a good sign neither result was a fluke. Nothing here looks suspiciously perfect (no single feature dominates near-totally), which is what I'd expect from a real signal rather than leakage.

**Where it's wrong:** looking at the top-50 ranked by the forest's probability, the false positives cluster in a specific pattern — pages with very poor `avg_position` (20–39, deep in the results) and near-zero CTR, that the model reads as high-risk based on position and CTR alone, but that are actually trending `stable` or even `up`. My read: the model is leaning on "this page performs badly" as a proxy for "this page is declining," and those aren't the same thing — a page can be a *permanently* weak performer without currently trending down. That's a real limitation worth carrying into the capstone, not just a curiosity.

In [5]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Random forest feature importances:")
print(importances.round(3))

print("\n--- 3 concrete wrong cases (top-50 by model score, but not actually declining) ---")
test_df = df.iloc[test_idx].copy()
test_df["rf_proba"] = rf.predict_proba(X_te)[:, 1]
top50 = test_df.sort_values("rf_proba", ascending=False).head(50)
wrong = top50[top50["is_declining_label"] == 0]
print(f"{len(wrong)} of the top 50 are false positives\n")
cols = ["content_type", "trend_direction", "avg_position", "ctr", "impressions_90d",
        "content_age_days", "rf_proba"]
wrong[cols].head(3)

Random forest feature importances:
impressions_90d           0.256
avg_position              0.233
content_age_days          0.156
word_count                0.145
ctr                       0.111
engagement_rate           0.063
days_since_last_update    0.037
dtype: float64

--- 3 concrete wrong cases (top-50 by model score, but not actually declining) ---


15 of the top 50 are false positives



,content_type,trend_direction,avg_position,ctr,impressions_90d,content_age_days,rf_proba
22526,keyword article,up,39.0,0.09,3445,280,0.976667
17451,comparison article,stable,8.5,0.00,1064,172,0.956667
4480,keyword article,stable,31.8,0.03,3171,280,0.953333


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.